<a href="https://colab.research.google.com/github/SaimaheshGangisetty/Zomato-Restaurant-Clustering-Unsupervised-ML/blob/main/Zomato_Restaurant_Clustering_Unsupervised_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Zomato Restaurant Clustering Unsupervised ML



##### **Project Type**    - Unsupervised
##### **Contribution**    - Individual
##### **Name**     - Sai Mahesh Gangisetty



# **Project Summary -**


In this project, restaurant review and metadata were analyzed
to understand customer behavior and satisfaction.

The dataset was cleaned and preprocessed to handle missing values,
outliers, and textual noise. Exploratory data analysis and
data visualization were performed using multiple charts
to identify important patterns and trends.

Sentiment analysis was applied to customer reviews
to understand overall opinions. Unsupervised learning techniques
such as KMeans, Hierarchical Clustering, and DBSCAN were used
to segment customers based on their behavior.

Model performance was evaluated using Silhouette Score
and hyperparameter tuning was performed to improve clustering quality.
The final results provide useful insights
for business decision-making and customer targeting.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


This project aims to analyze restaurant reviews and customer data
to understand customer sentiment and segment users
using unsupervised machine learning techniques.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
# Importing necessary libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

from textblob import TextBlob

import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('wordnet')


### Dataset Loading

In [ ]:
# Loading datasets with error handling

try:
    reviews_df = pd.read_csv("/content/Zomato Restaurant reviews.csv")
    restaurants_df = pd.read_csv("/content/Zomato Restaurant names and Metadata.csv")
    print("Datasets loaded successfully")
except:
    print("Error: Please upload dataset files")


### Dataset First View

In [ ]:

# Dataset first look
reviews_df.head()

In [ ]:
restaurants_df.head()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

print("Reviews Dataset:", reviews_df.shape)
print("Restaurants Dataset:", restaurants_df.shape)


### Dataset Information

In [ ]:
reviews_df.info()


In [ ]:
restaurants_df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count

print("Duplicate Reviews:", reviews_df.duplicated().sum())
print("Duplicate Restaurants:", restaurants_df.duplicated().sum())


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count

reviews_df.isnull().sum()



In [ ]:
restaurants_df.isnull().sum()


In [ ]:
# Visualizing the missing values
# Reviews Dataset
plt.figure(figsize=(10,5))
sns.heatmap(reviews_df.isnull(), cbar=False)
plt.title("Missing Values - Reviews Dataset")
plt.show()


In [ ]:
# Missing values visualization
# Restaurants Dataset

plt.figure(figsize=(10,5))
sns.heatmap(restaurants_df.isnull(), cbar=False)
plt.title("Missing Values - Restaurants Dataset")
plt.show()


### What did you know about your dataset?

- The reviews dataset contains textual reviews, ratings, and reviewer information.
- The restaurants dataset contains cost, cuisines, and operational details.
- Few missing values are present in metadata columns.
- Duplicate records are minimal.
- Both datasets can be merged using restaurant names.
- The dataset is suitable for clustering and segmentation analysis.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
reviews_df.columns


In [ ]:
restaurants_df.columns


In [ ]:
# Dataset Describe
# Statistical summary

reviews_df.describe()


In [ ]:
restaurants_df.describe()


### Variables Description


#### Reviews Dataset

- Restaurant: Name of the restaurant
- Reviewer: Name of the customer
- Review: Textual feedback given by the customer
- Rating: Rating given by the customer (1 to 5 scale)
- Metadata: Information about number of reviews and followers
- Time: Date and time of review
- Pictures: Number of images uploaded by reviewer

#### Restaurants Dataset

- Name: Restaurant name
- Links: Zomato URL
- Cost: Average cost for two people
- Collections: Zomato category tags
- Cuisines: Types of cuisines served
- Timings: Opening and closing timings


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
# Checking unique values for reviews dataset

for col in reviews_df.columns:
    print(col, ":", reviews_df[col].nunique())
print("")

# Checking unique values for restaurants dataset

for col in restaurants_df.columns:
    print(col, ":", restaurants_df[col].nunique())


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
reviews_df['Restaurant'] = reviews_df['Restaurant'].astype(str).str.strip().str.lower()
restaurants_df['Name'] = restaurants_df['Name'].astype(str).str.strip().str.lower()

In [ ]:
# Renaming column for merging

restaurants_df.rename(columns={'Name':'Restaurant'}, inplace=True)


In [ ]:
# Merging reviews and restaurants datasets

merged_df = pd.merge(
    reviews_df,
    restaurants_df,
    on='Restaurant',
    how='inner'
)

print("Merged Dataset Shape:", merged_df.shape)


In [ ]:
# Checking missing values after merging

merged_df.isnull().sum()


In [ ]:
# Removing commas and converting Cost to numeric (Clean Cost Column)

merged_df['Cost'] = merged_df['Cost'].astype(str).str.replace(',', '')
merged_df['Cost'] = pd.to_numeric(merged_df['Cost'], errors='coerce')


In [ ]:
# Filling missing cost with median

merged_df['Cost'] = merged_df['Cost'].fillna(merged_df['Cost'].median())


In [ ]:
# Filling missing values

merged_df['Review'] = merged_df['Review'].fillna("No Review")

merged_df['Collections'] = merged_df['Collections'].fillna("Not Available")

merged_df['Timings'] = merged_df['Timings'].fillna("Not Available")


In [ ]:
# Final dataset information

merged_df.info()


### What all manipulations have you done and insights you found?

### Data Wrangling and Cleaning

- Removed extra spaces and standardized restaurant names for accurate merging.
- Converted the Cost column from text to numeric format by removing commas.
- Merged reviews and restaurant datasets using restaurant name.
- Filled missing values using median and default text values.
- Removed duplicate records to improve data quality.
- Ensured that all important columns contain valid values.
- Prepared a clean and structured dataset for further analysis and modeling.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(8,5))
sns.countplot(x='Rating', data=merged_df)
plt.title("Distribution of Customer Ratings")
plt.show()


##### 1. Why did you pick the specific chart?

To understand how customers rate restaurants overall on zomato.

##### 2. What is/are the insight(s) found from the chart?

- Most customers have given high ratings, especially 4 and 5.
- Very few customers have given low ratings.
- This indicates that majority of restaurants provide satisfactory service.
- Some irregular values like "Like" and decimal ratings are also present.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### Will the gained insights help create a positive business impact?

Yes, this chart shows that most customers give high ratings. This helps in building trust and attracts more customers. Highly rated restaurants can be promoted to increase sales.

#### Are there any insights that may lead to negative growth?

Some restaurants have low ratings, which may reduce customer visits. Also, inconsistent values like "Like" in ratings can affect data quality. These issues should be improved to avoid negative impact.


#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(8,5))
sns.histplot(merged_df['Cost'], bins=20, kde=True)
plt.title("Distribution of Restaurant Cost")
plt.xlabel("Cost for Two")
plt.ylabel("Frequency")
plt.show()


##### 1. Why did you pick the specific chart?

To understand how restaurant prices are distributed and to identify low, medium, and high-cost restaurants.


##### 2. What is/are the insight(s) found from the chart?

- Most restaurants fall between ₹400 and ₹1000.
- Very few restaurants are extremely expensive.
- Medium-priced restaurants are more common.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, this chart helps in understanding the common price range of restaurants, which allows customers to choose places according to their budget. Medium-priced restaurants can attract more customers and increase sales. However, very expensive restaurants may receive fewer visits if customers feel they are not worth the price, which can lead to negative growth.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
# Top 10 most reviewed restaurants

top_restaurants = merged_df['Restaurant'].value_counts().head(10)

plt.figure(figsize=(10,5))
top_restaurants.plot(kind='bar')
plt.title("Top 10 Most Reviewed Restaurants")
plt.xlabel("Restaurant")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=45)
plt.show()


##### 1. Why did you pick the specific chart?

To identify the most popular restaurants based on the number of customer reviews.

##### 2. What is/are the insight(s) found from the chart?

- The top 10 restaurants have almost similar number of reviews.
- This shows that these restaurants are equally popular among customers.
- They have strong customer engagement on the platform.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, popular restaurants can be promoted to attract more users and increase revenue. However, less reviewed restaurants may get lower visibility, which can affect their growth if not improved.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
# Converting Rating to numeric (important for scatter plot)

merged_df['Rating_num'] = pd.to_numeric(merged_df['Rating'], errors='coerce')

plt.figure(figsize=(8,5))
sns.scatterplot(x='Cost', y='Rating_num', data=merged_df)

plt.title("Cost vs Rating")
plt.xlabel("Cost for Two")
plt.ylabel("Rating")
plt.show()


##### 1. Why did you pick the specific chart?

To understand the relationship between restaurant cost and customer ratings.

##### 2. What is/are the insight(s) found from the chart?

- High ratings are present across all price ranges.
- Both low-cost and high-cost restaurants receive good and bad ratings.
- There is no strong direct relationship between cost and rating.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, this shows that customers prefer quality over price. Affordable restaurants with good service can perform well. However, expensive restaurants may lose customers if they do not provide better quality, which can affect their growth.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
# Top 10 most popular cuisines

top_cuisines = merged_df['Cuisines'].value_counts().head(10)

plt.figure(figsize=(10,5))
top_cuisines.plot(kind='barh')

plt.title("Top 10 Most Popular Cuisines")
plt.xlabel("Number of Restaurants")
plt.ylabel("Cuisine")
plt.gca().invert_yaxis()   # To show highest on top

plt.show()


##### 1. Why did you pick the specific chart?

To identify the most preferred cuisines among customers.

##### 2. What is/are the insight(s) found from the chart?

- North Indian and Chinese cuisine is the most popular.
- Combination cuisines are more common than single cuisines.
- Desserts and ice cream also have good demand.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, restaurants can focus on popular cuisines to attract more customers. Introducing trending food combinations can increase sales. However, less popular cuisines may get fewer orders, which can affect their growth.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
# Taking top 8 cuisines for better visualization

top8_cuisines = merged_df['Cuisines'].value_counts().head(8).index

filtered_df = merged_df[merged_df['Cuisines'].isin(top8_cuisines)]

plt.figure(figsize=(12,6))

sns.boxplot(x='Cuisines', y='Cost', data=filtered_df)

plt.title("Cost Distribution Across Popular Cuisines")
plt.xlabel("Cuisine")
plt.ylabel("Cost for Two")

plt.xticks(rotation=45)
plt.show()


##### 1. Why did you pick the specific chart?

To compare the price range of different cuisines and understand which cuisines are more expensive or affordable.

##### 2. What is/are the insight(s) found from the chart?

- Combination cuisines like North Indian, Chinese, and Continental are more expensive.
- Ice Cream and Desserts are the most affordable.
- North Indian cuisine falls in the medium price range.
- Some cuisines show high variation in cost.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, restaurants can use this insight to design pricing strategies based on cuisine type. Affordable cuisines can attract more customers, while premium cuisines can target high-end users. However, very high prices may reduce customer interest if quality is not maintained.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
# Average rating for top 8 cuisines

avg_rating_cuisine = (
    merged_df[merged_df['Cuisines'].isin(top8_cuisines)]
    .groupby('Cuisines')['Rating_num']
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10,5))

avg_rating_cuisine.plot(kind='bar')

plt.title("Average Rating by Cuisine")
plt.xlabel("Cuisine")
plt.ylabel("Average Rating")

plt.xticks(rotation=45)
plt.show()


##### 1. Why did you pick the specific chart?

To compare the average customer ratings for different cuisines.
It helps in understanding which type of food is most liked by customers and which cuisines receive lower ratings.

##### 2. What is/are the insight(s) found from the chart?

- Ice Cream & Desserts has the highest average rating, showing strong customer satisfaction.

- Biryani and North Indian combinations also receive high ratings.

- Continental and pure North Indian cuisines have moderately good ratings.

- North Indian–Chinese–Biryani has the lowest average rating among the top cuisines.

This shows that customers prefer dessert and mixed-cuisine restaurants more than some traditional combinations.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato and restaurant owners focus more on high-rated cuisines for promotions.  
Low-rated cuisines indicate areas where food quality and service need improvement.

If ignored, these may lead to customer dissatisfaction and negative business growth.  
Improving them can increase customer trust, satisfaction, and revenue.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
# Chart 8: Cost vs Rating colored by Cuisine (Multivariate Analysis)

plt.figure(figsize=(10,6))

# Taking only top 5 cuisines to avoid overcrowding
top5 = merged_df['Cuisines'].value_counts().head(5).index

sns.scatterplot(
    data=merged_df[merged_df['Cuisines'].isin(top5)],
    x='Cost',
    y='Rating_num',
    hue='Cuisines'
)

plt.title("Cost vs Rating by Cuisine (Top 5)")
plt.xlabel("Cost for Two")
plt.ylabel("Rating")

plt.legend(bbox_to_anchor=(1.05,1))
plt.show()


##### 1. Why did you pick the specific chart?

To analyze the relationship between cost, rating, and cuisine type.  
It helps in understanding how different cuisines perform in terms of pricing and customer satisfaction at the same time.


##### 2. What is/are the insight(s) found from the chart?

- Ice Cream & Desserts has good ratings even at lower prices.
- North Indian cuisine shows moderate ratings at medium cost.
- North Indian–Chinese–Continental restaurants are mostly high-priced and receive mixed ratings.
- Some high-cost restaurants do not always receive high ratings.
- Lower and medium cost restaurants can also achieve good ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato and restaurant owners in pricing strategy.

- Affordable restaurants with good ratings can be promoted more.
- High-cost restaurants with low ratings should improve service and food quality.
- Customers can be guided towards value-for-money restaurants.

Ignoring this may lead to customer dissatisfaction and loss of trust.  
Using this information properly can increase customer retention and revenue.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
# Reviews Over Time

# Convert Time column to datetime
merged_df['Time'] = pd.to_datetime(merged_df['Time'], errors='coerce')

# Extract Year-Month
merged_df['YearMonth'] = merged_df['Time'].dt.to_period('M')

# Count reviews per month
reviews_time = merged_df.groupby('YearMonth').size()

plt.figure(figsize=(12,5))

reviews_time.plot(kind='line', marker='o')

plt.title("Number of Reviews Over Time")
plt.xlabel("Year-Month")
plt.ylabel("Number of Reviews")

plt.grid(True)
plt.show()


##### 1. Why did you pick the specific chart?

To analyze customer engagement trends over time.  
It helps in understanding how review activity changes month by month.


##### 2. What is/are the insight(s) found from the chart?

- Review activity was low during 2016 and 2017.
- A sharp increase is observed from mid-2018 onwards.
- The highest number of reviews appears in late 2018 and early 2019.
- This shows growing user engagement with the platform.

Overall, customer participation increased significantly over time.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato understand growth patterns.

- Increased reviews indicate rising customer trust and platform usage.
- High activity periods can be used for promotions and marketing campaigns.
- Low activity periods highlight the need for better engagement strategies.

If growth is not maintained, customer interest may decline.  
Using this data, Zomato can plan better marketing and retention strategies.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
# Reviewer Activity Level

# Extract number of reviews from Metadata column
# Example format: "3 Reviews , 2 Followers"

merged_df['Num_Reviews'] = merged_df['Metadata'].str.extract(r'(\d+)\s+Reviews').astype(float)

# Drop missing
reviewer_activity = merged_df.dropna(subset=['Num_Reviews'])

# Top 10 most active reviewers
top_reviewers = (
    reviewer_activity.groupby('Reviewer')['Num_Reviews']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10,5))

top_reviewers.plot(kind='bar')

plt.title("Top 10 Most Active Reviewers")
plt.xlabel("Reviewer")
plt.ylabel("Average Number of Reviews")

plt.xticks(rotation=60, ha='right')  # Rotate properly
plt.tight_layout()                  # Adjust spacing
plt.show()

##### 1. Why did you pick the specific chart?

To identify the most active reviewers on the Zomato platform.  
It helps in understanding which users contribute the most reviews and influence others.


##### 2. What is/are the insight(s) found from the chart?

- Anvesh Chowdary is the most active reviewer with the highest number of reviews.
- A few reviewers consistently contribute a large number of reviews.
- These users play an important role in shaping customer opinions.
- Most users contribute moderately, while only a few are highly active.
This shows that a small group of users generates most of the content.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato identify key influencers on the platform.

- Active reviewers can be rewarded or promoted to increase engagement.
- Their reviews can be highlighted to build customer trust.
- Engaging these users can improve platform credibility.

Ignoring top contributors may reduce content quality and user participation.  
Using this information can improve customer engagement and platform growth.


#### Chart - 11

In [ ]:
# Chart - 11 visualization code
# Followers vs Reviews

# Extract followers from Metadata
merged_df['Followers'] = merged_df['Metadata'].str.extract(r'(\d+)\s+Followers').astype(float)

# Drop missing values
influence_df = merged_df.dropna(subset=['Num_Reviews', 'Followers'])

plt.figure(figsize=(10,6))

plt.scatter(
    influence_df['Followers'],
    influence_df['Num_Reviews'],
    alpha=0.6
)

plt.title("Followers vs Number of Reviews")
plt.xlabel("Number of Followers")
plt.ylabel("Number of Reviews")

plt.grid(True)
plt.show()


##### 1. Why did you pick the specific chart?

To analyze the relationship between the number of followers and the number of reviews written by users.  
It helps in identifying influential reviewers on the platform.

##### 2. What is/are the insight(s) found from the chart?

- Most reviewers have low to moderate followers and write a moderate number of reviews.
- Some users with many followers do not necessarily write many reviews.
- A few users have both high followers and high review counts, showing strong influence.
- There is no perfect linear relationship between followers and reviews.

This shows that popularity does not always mean higher activity.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato identify real influencers.

- Users with both high followers and high activity can be targeted for promotions.
- Highly followed but inactive users need engagement strategies.
- Active reviewers with fewer followers can be promoted to increase visibility.

Ignoring this information may reduce platform engagement.  
Using it properly can improve trust, content quality, and user participation.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
# Review Length vs Rating

# Calculate review length (number of words)
merged_df['Review_Length'] = merged_df['Review'].apply(lambda x: len(str(x).split()))

# Remove missing ratings
length_df = merged_df.dropna(subset=['Rating_num'])

plt.figure(figsize=(10,6))

plt.scatter(
    length_df['Review_Length'],
    length_df['Rating_num'],
    alpha=0.5
)

plt.title("Review Length vs Rating")
plt.xlabel("Number of Words in Review")
plt.ylabel("Rating")

plt.grid(True)
plt.show()


##### 1. Why did you pick the specific chart?

To study the relationship between review length and customer ratings.  
It helps in understanding how much customers write when they are satisfied or dissatisfied.

##### 2. What is/are the insight(s) found from the chart?

- Most reviews are short and fall below 200 words.
- Both high and low ratings can have short reviews.
- Very long reviews are less common and mostly appear in extreme ratings.
- Customers tend to write longer reviews when they have strong opinions.

This shows that emotional experiences lead to longer feedback.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps in identifying customer sentiments.

- Long negative reviews may indicate serious complaints.
- Long positive reviews can be used for promotions.
- Short reviews may lack detailed feedback.

By analyzing long reviews, Zomato can quickly identify service issues and improve customer satisfaction.  
This helps in building trust and improving platform quality.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
# Pictures vs Rating

# Remove missing ratings
pic_df = merged_df.dropna(subset=['Rating_num'])

plt.figure(figsize=(10,6))

plt.scatter(
    pic_df['Pictures'],
    pic_df['Rating_num'],
    alpha=0.5
)

plt.title("Number of Pictures vs Rating")
plt.xlabel("Number of Pictures Uploaded")
plt.ylabel("Rating")

plt.grid(True)
plt.show()


##### 1. Why did you pick the specific chart?

To analyze the relationship between the number of pictures uploaded and customer ratings.  
It helps in understanding how customer engagement relates to satisfaction level.


##### 2. What is/are the insight(s) found from the chart?

- Most users upload very few or no pictures.
- Higher ratings are often associated with more pictures.
- Users who upload many pictures usually have positive experiences.
- Very few users upload many pictures with low ratings.

This shows that satisfied customers are more likely to share photos.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps Zomato improve platform credibility.

- Photo-rich reviews increase trust among customers.
- Highly engaged users can be rewarded to encourage more content.
- Restaurants with more photos appear more attractive to users.

Ignoring visual content may reduce user interest.  
Encouraging photo uploads can increase engagement and conversions.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

# Select only numeric columns
numeric_df = merged_df[['Rating_num', 'Cost', 'Pictures', 'Review_Length',
                         'Num_Reviews', 'Followers']]

# Calculate correlation matrix
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numeric Features")
plt.show()


##### 1. Why did you pick the specific chart?

To analyze the relationship between different numeric variables in the dataset.  
It helps in understanding how features are related to each other and supports feature selection for machine learning models.


##### 2. What is/are the insight(s) found from the chart?

- Rating has very low correlation with cost, pictures, and review length.
- Pictures and review length show moderate positive correlation.
- Number of reviews and followers are moderately correlated.
- Cost has very weak correlation with most features.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Select important numeric features
pairplot_data = merged_df[['Rating_num', 'Cost', 'Pictures',
                           'Review_Length', 'Num_Reviews', 'Followers']]

# Create Pair Plot
sns.pairplot(pairplot_data, diag_kind='kde')

plt.suptitle("Pair Plot of Important Numeric Features", y=1.02)
plt.show()


##### 1. Why did you pick the specific chart?

To visualize the relationships between multiple numeric variables at once.  
It helps in identifying patterns, trends, and possible correlations in the dataset.

##### 2. What is/are the insight(s) found from the chart?

- Most features show weak to moderate relationships.
- Rating does not strongly depend on cost, pictures, or review length.
- Review length and number of reviews show a positive relationship.
- Number of reviews and followers are positively related.
- Cost does not directly affect customer ratings.

This shows that customer engagement matters more than pricing.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Relationship between Cost and Rating

#### 1. Null and Alternative Hypothesis

  **Null Hypothesis (H0):**  
There is no significant relationship between restaurant cost and customer rating.

**Alternative Hypothesis (H1):**  
There is a significant relationship between restaurant cost and customer rating.


#### 2. Perform an appropriate statistical test.

In [ ]:
merged_df[['Cost', 'Rating_num']].isnull().sum()


In [ ]:
# Drop rows with missing Cost or Rating
clean_df1 = merged_df[['Cost', 'Rating_num']].dropna()

from scipy.stats import pearsonr

# Run Pearson on cleaned data
corr1, pval1 = pearsonr(clean_df1['Cost'], clean_df1['Rating_num'])

print("Correlation:", corr1)
print("P-value:", pval1)


##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Test

##### Why did you choose the specific statistical test?

Pearson correlation is suitable because both cost and rating are continuous numerical variables.  
It measures the linear relationship between them.

##### Inference Based on Test Result
Since the p-value is less than 0.05, the null hypothesis is rejected.  
This indicates that cost has a weak but statistically significant effect on ratings.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Relationship between Number of Reviews and Followers

#### 1. State Your Research Hypothesis

**Null Hypothesis (H0):**  
There is no significant relationship between number of reviews and followers.

**Alternative Hypothesis (H1):**  
There is a significant relationship between number of reviews and followers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

# Pearson Correlation between Number of Reviews and Followers

# Remove missing values
clean_df2 = merged_df[['Num_Reviews', 'Followers']].dropna()

# Apply Pearson Correlation
corr2, pval2 = pearsonr(clean_df2['Num_Reviews'], clean_df2['Followers'])

print("Correlation:", corr2)
print("P-value:", pval2)


##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Test


##### Why did you choose the specific statistical test?

Number of reviews and followers are continuous numerical variables.  
Pearson correlation is suitable to measure their relationship.


##### Inference Based on Test Result
Since the p-value is less than 0.05, the null hypothesis is rejected.  
This indicates a statistically significant and moderate positive relationship between number of reviews and followers.

This means that users who write more reviews tend to gain more followers.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Relationship between Review Length and Rating

**Null Hypothesis (H0):**  
There is no significant relationship between review length and customer rating.

**Alternative Hypothesis (H1):**  
There is a significant relationship between review length and customer rating.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Pearson Correlation between Review Length and Rating


# Remove missing values
clean_df3 = merged_df[['Review_Length', 'Rating_num']].dropna()

# Apply Pearson Correlation
corr3, pval3 = pearsonr(clean_df3['Review_Length'], clean_df3['Rating_num'])

print("Correlation:", corr3)
print("P-value:", pval3)


##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Test


##### Why did you choose the specific statistical test?

Review length and rating are continuous numerical variables.  
Pearson correlation is suitable to measure their relationship.

##### Inference Based on Test Result
Since the p-value is less than 0.05, the null hypothesis is rejected.  
This indicates a statistically significant but very weak negative relationship between review length and rating.

This suggests that longer reviews are slightly associated with lower ratings, which may represent customer complaints or detailed feedback.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

# Handling Missing Values

# Fill categorical missing values
merged_df['Reviewer'] = merged_df['Reviewer'].fillna("Unknown")
merged_df['Metadata'] = merged_df['Metadata'].fillna("Not Available")
merged_df['Collections'] = merged_df['Collections'].fillna("Not Available")

# Fill numerical missing values
merged_df['Rating_num'] = merged_df['Rating_num'].fillna(merged_df['Rating_num'].median())

# Fill Time using mode
merged_df['Time'] = merged_df['Time'].fillna(merged_df['Time'].mode()[0])

# Check again
merged_df.isnull().sum()


#### What all missing value imputation techniques have you used and why did you use those techniques?

Missing values were handled using median for numerical columns and meaningful labels for categorical columns.
This helps in reducing bias and maintaining data consistency.


### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Handling Outliers in Cost Column

Q1 = merged_df['Cost'].quantile(0.25)
Q3 = merged_df['Cost'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Remove outliers
merged_df = merged_df[(merged_df['Cost'] >= lower) & (merged_df['Cost'] <= upper)]

merged_df.shape


##### What all outlier treatment techniques have you used and why did you use those techniques?

Outliers in Cost were removed using the IQR method to reduce noise and improve model performance.


### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# One Hot Encoding for categorical features

encoded_df = pd.get_dummies(
    merged_df,
    columns=['Cuisines', 'Collections'],
    drop_first=True
)

encoded_df.head()


#### What all categorical encoding techniques have you used & why did you use those techniques?

One-Hot Encoding was used to convert categorical features into numerical format required for ML models.


### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand contractions manually

contraction_dict = {
    "don't": "do not",
    "can't": "cannot",
    "won't": "will not",
    "i'm": "i am",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "that's": "that is",
    "what's": "what is",
    "there's": "there is",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "shouldn't": "should not"
}

def expand_contractions(text):
    for word in contraction_dict:
        text = text.replace(word, contraction_dict[word])
    return text


merged_df['Review_Expanded'] = merged_df['Review'].str.lower().apply(expand_contractions)

merged_df[['Review','Review_Expanded']].head()


#### 2. Lower Casing

In [ ]:
# Lower Casing
# already done in the above step

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
import string

# Remove punctuation
merged_df['Review_NoPunc'] = merged_df['Review_Expanded'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

merged_df['Review_NoPunc'].head()


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
import re

# Remove URLs and digits
merged_df['Review_NoURL'] = merged_df['Review_NoPunc'].apply(
    lambda x: re.sub(r'http\S+|www\S+|\d+', '', x)
)

merged_df['Review_NoURL'].head()


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

merged_df['Review_NoStop'] = merged_df['Review_NoURL'].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)


In [ ]:
# Remove White spaces
merged_df['Review_NoStop'] = merged_df['Review_NoStop'].str.strip()

merged_df['Review_NoStop'].head()

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
merged_df['Clean_Review'] = merged_df['Review_NoStop']

merged_df['Clean_Review'].head()


#### 7. Tokenization

In [ ]:
# Tokenization
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

# Tokenize cleaned reviews
merged_df['Review_Tokens'] = merged_df['Clean_Review'].apply(
    lambda x: word_tokenize(x)
)

merged_df['Review_Tokens'].head()


#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
import nltk
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Lemmatize tokens
merged_df['Review_Lemma'] = merged_df['Review_Tokens'].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)

merged_df['Review_Lemma'].head()


##### Which text normalization technique have you used and why?

Lemmatization was used to convert words into their meaningful base form
while preserving context and reducing noise in the text.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
import nltk

nltk.download('averaged_perceptron_tagger_eng')

# POS Tagging (Fixed Version)

merged_df['POS_Tags'] = merged_df['Review_Lemma'].apply(
    lambda x: nltk.pos_tag(x)
)

merged_df['POS_Tags'].head()



In [ ]:

# Join lemmatized words into sentences
merged_df['Final_Text'] = merged_df['Review_Lemma'].apply(
    lambda x: " ".join(x)
)

print(merged_df['Final_Text'].head())


SENTIMENTAL ANALYSIS

In [ ]:
from textblob import TextBlob

# Function to calculate sentiment polarity
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

# Apply sentiment analysis
merged_df['Sentiment_Score'] = merged_df['Final_Text'].apply(get_sentiment)

merged_df[['Final_Text', 'Sentiment_Score']].head()


In [ ]:
# Convert score to sentiment category
def sentiment_label(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

merged_df['Sentiment_Label'] = merged_df['Sentiment_Score'].apply(sentiment_label)

merged_df['Sentiment_Label'].value_counts()


In [ ]:
import matplotlib.pyplot as plt

sentiment_counts = merged_df['Sentiment_Label'].value_counts()

plt.figure(figsize=(6,4))
sentiment_counts.plot(kind='bar')
plt.title("Sentiment Distribution of Customer Reviews")
plt.xlabel("Sentiment Type")
plt.ylabel("Number of Reviews")
plt.show()


Sentiment analysis was used to understand
customer opinions and satisfaction levels.

Clustering models were used to segment customers
based on behavioral and numerical features.

Combining both helped in identifying
valuable and satisfied customer groups.


#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=500,
    min_df=5,
    max_df=0.8
)

# Transform text into vectors
tfidf_matrix = tfidf.fit_transform(merged_df['Final_Text'])

tfidf_matrix.shape



##### Which text vectorization technique have you used and why?

TF-IDF vectorization was used to convert textual reviews into numerical features.
It assigns higher importance to meaningful words and reduces the impact of common words.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

# Log transform Cost to reduce skewness
merged_df['Log_Cost'] = np.log1p(merged_df['Cost'])

# Interaction Feature
merged_df['Cost_x_Rating'] = merged_df['Cost'] * merged_df['Rating_num']

merged_df[['Log_Cost','Cost_x_Rating']].head()

# Log transformation was applied on Cost to reduce skewness.
# Interaction features were created to capture combined effects.



#### 2. Feature Selection

In [ ]:
# Create Rating_per_Cost Feature

merged_df['Rating_per_Cost'] = merged_df['Rating_num'] / merged_df['Cost']

merged_df[['Rating_num','Cost','Rating_per_Cost']].head()


In [ ]:
# Feature Selection using Correlation

feature_corr = merged_df[
    ['Rating_num','Cost','Pictures',
     'Review_Length','Rating_per_Cost',
     'Log_Cost','Cost_x_Rating']
].corr()

feature_corr['Rating_num'].sort_values(ascending=False)


##### What all feature selection methods have you used  and why?

Correlation-based feature selection was used to identify relevant variables
that strongly influence customer ratings.


##### Which all features you found important and why?

Important features selected include Cost, Review Length, Rating per Cost,
and Pictures as they show strong relationship with customer ratings.


### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
# using Log Transform

merged_df['Transformed_Cost'] = np.log1p(merged_df['Cost'])

merged_df[['Cost','Transformed_Cost']].head()


Yes, data transformation was needed because the Cost feature was highly skewed
and had extreme values, which could affect model performance.

Log transformation was used to reduce skewness and balance the data,
making it more suitable for machine learning models.


### 6. Data Scaling

In [ ]:
# Scaling your data
from sklearn.preprocessing import StandardScaler

# Select numerical features for scaling
scale_features = merged_df[
    ['Cost','Pictures','Review_Length',
     'Rating_per_Cost','Log_Cost']
]

# Apply Standard Scaling
scaler = StandardScaler()

scaled_data = scaler.fit_transform(scale_features)

scaled_data.shape


##### Which method have you used to scale you data and why?


StandardScaler was used to scale numerical features so that all variables
have equal importance and are not dominated by large values.
This improves clustering performance.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes, dimensionality reduction is needed because the dataset contains multiple
numerical and encoded features, which may lead to high dimensionality and noise.
Reducing dimensions helps in better visualization and faster model execution.


In [ ]:
# DImensionality Reduction (If needed)
from sklearn.decomposition import PCA

# Apply PCA
pca = PCA(n_components=2)

pca_data = pca.fit_transform(scaled_data)

pca_data.shape


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Dimensionality reduction was optional because the dataset contained only five numerical features. However, PCA was applied mainly for visualization and better interpretation of clusters, while the original scaled data was used for model training.


### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split

# Define features and target (for evaluation purpose)
X = scaled_data
y = merged_df['Rating_num']

# Split data (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Check shapes
X_train.shape, X_test.shape


##### What data splitting ratio have you used and why?

An 80:20 data splitting ratio was used, where 80% of the data was used for training
and 20% for testing. This ensures good learning and reliable evaluation.


### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.



Yes, the dataset shows moderate imbalance in customer ratings, as certain
rating values occur more frequently than others.
This is evident from the rating distribution analysis.


# Check rating distribution


In [ ]:
# Handling Imbalanced Dataset (If needed)
# Check rating distribution
rating_dist = merged_df['Rating_num'].value_counts(normalize=True)

rating_dist


Yes, the dataset is moderately imbalanced, as higher ratings like 5.0 (38%)
and 4.0 (24%) occur much more frequently than lower ratings.
Lower and half-star ratings are very rare.


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

No imbalance handling technique was applied because this is an unsupervised
learning problem. Clustering algorithms do not depend on class labels,
so class imbalance does not directly affect model performance.


## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

from sklearn.cluster import KMeans

# Apply KMeans with initial K = 3
kmeans = KMeans(n_clusters=3, random_state=42)

# Fit and predict
kmeans_labels = kmeans.fit_predict(scaled_data)

# Store labels
merged_df['KMeans_Cluster'] = kmeans_labels

# Check cluster counts
merged_df['KMeans_Cluster'].value_counts()


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
from sklearn.metrics import silhouette_score

kmeans_sil = silhouette_score(scaled_data, kmeans_labels)

print( "Silhouette score:",kmeans_sil)

import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.scatter(
    pca_data[:,0],
    pca_data[:,1],
    c=kmeans_labels,
    cmap='viridis'
)

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('KMeans Clustering Visualization')
plt.colorbar(label='Cluster')
plt.show()


K-Means clustering was used to group similar restaurants and reviews
based on numerical and textual features.

PCA was used only for visualization.

Silhouette Score was used to evaluate cluster quality.
Higher values indicate better clustering.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques

sil_scores = []

K_range = range(2, 9)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(scaled_data)

    score = silhouette_score(scaled_data, labels)
    sil_scores.append(score)

# Plot Silhouette Scores
plt.figure(figsize=(7,5))
plt.plot(K_range, sil_scores, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs K')
plt.show()

sil_scores


##### Which hyperparameter optimization technique have you used and why?

Manual Grid Search was done by testing different K values (2–8)
and selecting the best one using Silhouette Score.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, improvement was observed.
After tuning, K = 5 gave the highest Silhouette Score (0.408),
resulting in better cluster separation.


### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering

# Dendrogram
Z = linkage(scaled_data, method='ward')

plt.figure(figsize=(12,6))
dendrogram(Z)
plt.title("Dendrogram for Hierarchical Clustering")
plt.xlabel("Data Points")
plt.ylabel("Distance")
plt.show()

# Initial hierarchical model (for analysis)
hierarchical_model = AgglomerativeClustering(
    n_clusters=5,
    linkage='ward'
)

hierarchical_labels = hierarchical_model.fit_predict(scaled_data)

# Store clusters
merged_df['Hierarchical_Cluster'] = hierarchical_labels


The dendrogram shows how data points are grouped step by step based on similarity.
By observing the largest vertical distance, around 3–5 clusters can be identified.
Hence, 5 clusters were selected for Hierarchical Clustering.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2  Implementation with hyperparameter optimization techniques


from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

linkages = ['ward', 'complete', 'average']
n_clusters_list = [2, 3, 4, 5, 6]

results = []

for link in linkages:
    for k in n_clusters_list:

        model = AgglomerativeClustering(
            n_clusters=k,
            linkage=link
        )

        labels = model.fit_predict(scaled_data)
        score = silhouette_score(scaled_data, labels)

        results.append((link, k, score))

        print(f"Linkage: {link}, K: {k}, Silhouette: {score}")




##### Which hyperparameter optimization technique have you used and why?

Manual hyperparameter tuning was used by testing different linkage methods
(ward, complete, average) and different numbers of clusters.
Silhouette Score was used to compare all combinations
and select the best performing configuration.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, the Silhouette Score increased from around 0.39 (ward linkage)
to about 0.86 using complete and average linkage with 2 clusters.



#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Silhouette Score was used as the main evaluation metric.
It measures how well data points fit within their clusters and how well clusters are separated.

A higher score indicates clear customer segmentation and better identification of similar users.
This helps in targeted marketing, improved engagement, and higher customer retention.

Low scores may lead to unclear segmentation and ineffective business strategies.


### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Apply DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)

dbscan_labels = dbscan.fit_predict(scaled_data)

# Store labels
merged_df['DBSCAN_Cluster'] = dbscan_labels

# Check cluster counts
merged_df['DBSCAN_Cluster'].value_counts()


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
# Remove noise points (-1)
mask = dbscan_labels != -1

dbscan_score = silhouette_score(
    scaled_data[mask],
    dbscan_labels[mask]
)

print("DBSCAN Silhouette Score:", dbscan_score)

# Visualization
plt.figure(figsize=(8,6))

plt.scatter(
    pca_data[:,0],
    pca_data[:,1],
    c=dbscan_labels,
    cmap='viridis'
)

plt.title("DBSCAN Clustering Visualization")
plt.colorbar()
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Try different values of eps
eps_values = [0.3, 0.5, 0.7, 1.0]
dbscan_scores = []

for eps in eps_values:

    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(scaled_data)

    # Remove noise points
    mask = labels != -1

    # Calculate Silhouette only if more than 1 cluster exists
    if len(set(labels)) > 1:
        score = silhouette_score(scaled_data[mask], labels[mask])
        dbscan_scores.append(score)
    else:
        dbscan_scores.append(None)

    print(f"eps = {eps}, Silhouette Score = {score}")


##### Which hyperparameter optimization technique have you used and why?

Manual hyperparameter tuning was used for DBSCAN by experimenting with different
values of the eps parameter.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, significant improvement was observed after hyperparameter tuning.
Initially, the Silhouette Score was low (0.19) at eps = 0.5.
After tuning, the best score (0.58) was achieved at eps = 1.0.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Silhouette Score was used as the main evaluation metric for all three models.
It measures how well data points fit within their clusters
and how well clusters are separated.
A higher Silhouette Score indicates better customer segmentation,
which helps in targeted marketing, personalization, and better decision-making.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Hierarchical Clustering was selected as the final model.

It achieved the highest Silhouette Score (around 0.86)
using complete and average linkage with 2 clusters.
Compared to KMeans and DBSCAN, it produced clearer
and more meaningful customer groups.
Therefore, it was chosen for better business insights.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Hierarchical Clustering groups customers based on similarity
using a bottom-up approach.

Each data point starts as an individual cluster and similar clusters
are merged step by step.

Correlation analysis, PCA, and cluster profiling
were used as model explainability tools.

These methods helped in understanding how features influence clustering.

Cost, Review Length, and Number of Reviews were found to have
the strongest impact on customer grouping.


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
import joblib

# Save the trained hierarchical model
joblib.dump(hierarchical_model, "hierarchical_model.pkl")

print("Model saved successfully!")


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
# Load the saved model
loaded_model = joblib.load("hierarchical_model.pkl")

print("Model loaded successfully!")

# Test on few samples
test_data = scaled_data[:5]

pred = loaded_model.fit_predict(test_data)

print("Predicted clusters:", pred)



### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

In this project, restaurant review data was analyzed
to understand customer behavior and satisfaction.

After cleaning and processing the data,
different patterns related to cost, ratings, and reviews were explored.
Sentiment analysis helped in knowing whether customers
were happy or unhappy with the restaurants.
Clustering models were then used to group similar customers.

These findings can help restaurant owners
improve their services, plan better offers,
and connect better with customers.

Overall, this project shows how data analysis
and machine learning can be used
to solve real-life business problems in a practical way.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***